# BML Sensitivity Analysis

This notebook sweeps the two main free parameters of the
**Biham–Middleton–Levine (BML)** model to understand how each affects flow:

| Section | Parameter swept | Fixed at |
|---|---|---|
| 1 | Horizontal car fraction $r_{\text{horiz}}$ | $\rho \approx \rho_c$, $L = 100$ |
| 2 | Grid size $L$ | $\rho \in [0.05, 0.65]$ |
| 3 | Summary | — |

The notebook is **self-contained**: the `BML` class is re-defined here so it can
be run independently of `biham_middleton_levine.ipynb`.

## Setup — Model & Parameters

In [ ]:
# ── Imports and global parameters ─────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

L         = 100   # default grid size
T_WARMUP  = 500   # transient steps discarded
T_MEASURE = 200   # steps averaged for flow

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})


class BML:
    '''
    Biham-Middleton-Levine 2-D traffic cellular automaton.

    Grid: L x L array (int8).  0 = empty, 1 = horizontal car, 2 = vertical car.
    Boundary: periodic (torus).

    One full timestep:
      Phase 1  - horizontal cars move right (if target empty)
      Phase 2  - vertical cars move up   (if target empty)
    Both phases are evaluated simultaneously on the state *before* any move
    in that phase is applied.
    '''

    def __init__(self, L, density, r_horiz=0.5, seed=None):
        self.L = L
        self.t = 0
        rng    = np.random.default_rng(seed)

        n_total = int(round(density * L * L))
        n_horiz = int(round(r_horiz * n_total))
        n_vert  = n_total - n_horiz

        flat      = rng.permutation(L * L)
        self.grid = np.zeros(L * L, dtype=np.int8)
        self.grid[flat[:n_horiz]]                  = 1
        self.grid[flat[n_horiz:n_horiz + n_vert]]  = 2
        self.grid = self.grid.reshape(L, L)

    def _move_horizontal(self):
        can_move = (self.grid == 1) & (np.roll(self.grid, -1, axis=1) == 0)
        n_moved  = int(can_move.sum())
        if n_moved:
            g = self.grid.copy()
            g[can_move] = 0
            g[np.roll(can_move, 1, axis=1)] = 1
            self.grid = g
        return n_moved

    def _move_vertical(self):
        can_move = (self.grid == 2) & (np.roll(self.grid, 1, axis=0) == 0)
        n_moved  = int(can_move.sum())
        if n_moved:
            g = self.grid.copy()
            g[can_move] = 0
            g[np.roll(can_move, -1, axis=0)] = 2
            self.grid = g
        return n_moved

    def step(self):
        n_total = int((self.grid > 0).sum())
        moved_h = self._move_horizontal()
        moved_v = self._move_vertical()
        self.t += 1
        return (moved_h + moved_v) / max(n_total, 1)

    def warmup(self, T=None):
        for _ in range(T or T_WARMUP):
            self.step()

    def run(self, T=None):
        T = T or T_MEASURE
        return float(np.mean([self.step() for _ in range(T)]))

    def density(self):
        return float((self.grid > 0).sum()) / self.L ** 2

    def is_gridlocked(self):
        h_free = ((self.grid == 1) & (np.roll(self.grid, -1, axis=1) == 0)).any()
        v_free = ((self.grid == 2) & (np.roll(self.grid,  1, axis=0) == 0)).any()
        return not (h_free or v_free)


def run_sweep(L, densities, seed_offset=0, T_warmup=T_WARMUP, T_measure=T_MEASURE):
    flows = np.zeros(len(densities))
    for i, rho in enumerate(densities):
        sim      = BML(L=L, density=rho, r_horiz=0.5, seed=seed_offset + i)
        sim.warmup(T_warmup)
        flows[i] = sim.run(T_measure)
    return flows


# Estimate rho_c for default L=100 (used throughout)
rhos_ref  = np.linspace(0.10, 0.60, 50)
phi_ref   = run_sweep(L=100, densities=rhos_ref)
dphi_ref  = -np.gradient(phi_ref, rhos_ref)
rho_c     = float(rhos_ref[np.argmax(dphi_ref)])
print(f'BML class defined.  Estimated \u03c1_c = {rho_c:.3f}  (L = 100)')

## Section 1: Effect of Car Type Ratio

The standard BML model uses equal car counts ($r_{\text{horiz}} = 0.5$).
Varying the ratio at fixed total density tests how **asymmetry** affects the transition:

- Near $\rho_c$, extreme ratios (many of one type, few of the other) give the **minority
  type** more freedom to move and may delay gridlock.
- Far below $\rho_c$, the ratio has little effect since both types flow freely.
- The symmetric case is the worst-case scenario for mutual blocking.

In [ ]:
# ── Section 1: Effect of car type ratio ───────────────────────────────────
rho_fixed = round(rho_c, 2)   # density near the critical point
ratios    = np.linspace(0.10, 0.90, 17)
flows_r   = []

for i, r in enumerate(ratios):
    sim = BML(L=100, density=rho_fixed, r_horiz=r, seed=42 + i)
    sim.warmup(T_WARMUP)
    flows_r.append(sim.run(T_MEASURE))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ratios, flows_r, 'o-', color='mediumpurple', lw=2, ms=6)
ax.axvline(0.5, color='grey', ls='--', lw=1.2, label='Symmetric  (r = 0.5)')
ax.set_xlabel('Horizontal car fraction  r_horiz')
ax.set_ylabel('Mean flow')
ax.set_title(f'Effect of Car Type Ratio  (\u03c1 = {rho_fixed}, L = 100)')
ax.set_xlim(0.05, 0.95)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 2: System Size Effects

The BML transition sharpens with $L$ because larger grids self-average better
over the random initial conditions.  Small systems may or may not jam at a given
$\rho$ depending on the specific realisation; large systems behave deterministically
relative to density.

We also observe that $\rho_c$ shifts slightly with $L$ (finite-size scaling).

In [ ]:
# ── Section 2: System size effects ────────────────────────────────────────
sizes_all  = [32, 64, 128, 256]
cols_all   = ['coral', 'steelblue', 'seagreen', 'goldenrod']
rhos_sz    = np.linspace(0.05, 0.65, 25)

fd_all = {}
fig, ax = plt.subplots(figsize=(8, 5))

for L_val, col in zip(sizes_all, cols_all):
    phi           = run_sweep(L=L_val, densities=rhos_sz)
    fd_all[L_val] = phi
    ax.plot(rhos_sz, phi, 'o-', color=col, lw=2, ms=4, label=f'L = {L_val}')

ax.set_xlabel('Density \u03c1')
ax.set_ylabel('Mean flow')
ax.set_title('Fundamental Diagram: System Size Effects')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 3: Summary

### Key Sensitivity Findings

1. **Car type ratio**: the symmetric case ($r_{\text{horiz}} = 0.5$) minimises flow near
   $\rho_c$ — mutual blocking is maximised when the two streams are equally matched.
   Extreme ratios restore partial flow for the minority type.

2. **System size**: larger $L$ produces a sharper, more abrupt transition.
   The estimated $\rho_c$ converges from above as $L \to \infty$, consistent
   with finite-size scaling theory.

In [ ]:
# ── Section 3: Summary ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ratio sweep
axes[0].plot(ratios, flows_r, 'o-', color='mediumpurple', lw=2, ms=6)
axes[0].axvline(0.5, color='grey', ls='--', lw=1.2, label='Symmetric')
axes[0].set_xlabel('Horizontal car fraction  r_horiz')
axes[0].set_ylabel('Mean flow')
axes[0].set_title(f'Car Type Ratio  (\u03c1 = {rho_fixed})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: all system sizes
for L_val, col in zip(sizes_all, cols_all):
    axes[1].plot(rhos_sz, fd_all[L_val], '-', color=col, lw=2.5, label=f'L = {L_val}')
axes[1].set_xlabel('Density \u03c1')
axes[1].set_ylabel('Mean flow')
axes[1].set_title('System Size Effects')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('BML Sensitivity Analysis — Summary', fontsize=13)
plt.tight_layout()
plt.show()